# D3: block ownership, reference counts and reuse

In D2, we manually chose which physical KV blocks belonged to each request. D3 adds the bookkeeping that makes those choices for us. A **request** is one independent token sequence. A **physical block** is a fixed-size region of the shared key/value cache.

We will follow the complete lifetime of a request: reserve blocks, process tokens, reserve more space, handle a failed forward, and release blocks so another request can reuse them.

**What must stay correct?** Live requests must not accidentally receive the same block. A block becomes free only when its reference count reaches zero. A reservation that cannot fit must leave ownership unchanged. Reserving space must not claim that tokens have already been processed. Reusing old storage must produce the same logits as fresh reference forwards.

There are two examples in this lesson. The small allocator example explains reference counts, including multiple references to one block. The model example uses **disjoint blocks**: each assigned block belongs to exactly one request. Shared-prefix attention and copy-on-write are not implemented. A scheduler that decides which requests run, preemption, and fast attention kernels also come later. D2 manual tables and existing no-cache/contiguous-cache paths remain supported.

Run top to bottom on a **fresh Colab T4**. Two FP32 TinyLlama copies need about 8.8 GB of weights. The complete November 4, 2026 Colab run passed 253 tests with one two-GPU test skipped, including failure/retry, growth, reuse and reset. Its original evidence is archived in [D3_block_manager.json](../results/D3_block_manager.json). Local checks remain static only. Preserve previous outputs before replacing the runtime.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/guptasameer112/tinyserve.git"
REPO_DIR = "/content/tinyserve"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], check=True, capture_output=True, text=True).stdout)


## 1. Test first

The next cell runs the B8/C2/C3/D1/D2 tests as regression checks, then the new D3 tests. This checks that adding block ownership does not break the existing model, generation or cache paths.

D3 tests both individual operations and longer sequences of operations. Twenty seeded allocator traces each perform 400 allocations, retains and releases. A separate counter model checks the expected reference counts after every step. Ten managed-request traces each perform 200 reservations and releases, checking that live request tables never overlap and that every physical block is either assigned or free. A fixed seed makes a failing sequence reproducible.

Boundary tests cover rounding token capacity to whole blocks, insufficient free space, invalid input, and the difference between reset and release. Small CPU/CUDA model tests compare managed cached logits with separate no-cache forwards from our model and Hugging Face. They also deliberately fail a later layer, then retry the same tokens.

A failed test stops the lesson rather than continuing to the demo. The C2 test that requires two GPUs may skip on a single T4. The CUDA tests should run here.

In [ ]:
import gc
import torch
from tinyserve.basics import load_model, encode
from tinyserve.engine.block_manager import BlockAllocator, BlockPoolExhausted
from tinyserve.models.llama import LlamaForCausalLM
from tinyserve.models.attention import AttentionMetadata
from tinyserve.probe import compare_paged_batch
from tinyserve.envinfo import collect_env, save_result

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then rerun.")
DEVICE = "cuda"
torch.backends.cuda.matmul.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
print(collect_env())
test_run = subprocess.run([sys.executable, "-m", "pytest", "tests/test_b8_model.py",
                           "tests/test_c2_generation.py", "tests/test_c3_cache.py",
                           "tests/test_d1_batching.py", "tests/test_d2_paged_cache.py",
                           "tests/test_d3_block_manager.py", "-v", "-ra"],
                          capture_output=True, text=True)
print(test_run.stdout)
print(test_run.stderr)
test_run.check_returncode()


## 2. What does a reference count mean?

A **reference** is one ownership claim that says: "I still need this block, so do not reuse it." We previously called this a lease. There is no timer involved, and a reference is not a token or a copy of the K/V data.

Two owners can refer to the same physical block without creating a second copy of its data:

```
owner A ----+
           +---- physical block 0 (one stored K/V block)
owner B ----+
```

The allocator tracks the number of claims, not the owners' identities. `allocate(2)` returns two free block IDs and gives each an initial count of one. `retain(0)` adds another claim on block 0, changing its count from one to two. `release(0)` removes one claim. Every allocation or retain must eventually have a matching release.

Follow the next cell with this table. Counts are ordered by physical block ID: 0, 1, 2.

| Operation | Reference counts | Free IDs, next allocation first |
|---|---|---|
| Start | `[0, 0, 0]` | `[0, 1, 2]` |
| Allocate blocks 0 and 1 | `[1, 1, 0]` | `[2]` |
| Retain block 0 | `[2, 1, 0]` | `[2]` |
| Release block 0 once | `[1, 1, 0]` | `[2]` |
| Release block 0 again | `[0, 1, 0]` | `[0, 2]` |
| Allocate one block: reuse 0 | `[1, 1, 0]` | `[2]` |

The first release cannot free block 0 because one claim remains. The second release reaches zero, so the block becomes available for reuse. Our free list is a stack: release pushes an ID onto the end, and allocation pops from that end. This makes recently released block 0 come out before block 2, which was free longer.

The conservation rule counts **physical blocks**, not references: `free_count + live_physical_block_count == pool_size`. After retaining block 0, there are two live blocks but three references. Those are different quantities.

This allocator example explains the lifetime rule only. Actual D3 model requests do not share blocks. Reference counts alone would not make independent writes to a shared block safe; that would also require shared-prefix rules and copy-on-write.

In [ ]:
allocator = BlockAllocator(3)
allocator_trace = []

def record_allocator(label):
    assert set(allocator.free_blocks) == {index for index, refs in enumerate(allocator.ref_counts) if refs == 0}
    assert allocator.num_free_blocks + sum(refs > 0 for refs in allocator.ref_counts) == allocator.num_blocks
    row = {"label": label, "free_blocks": list(allocator.free_blocks), "ref_counts": list(allocator.ref_counts)}
    allocator_trace.append(row)
    print(row)

record_allocator("start")
assert allocator.allocate(2) == (0, 1)
record_allocator("allocate two")
allocator.retain(0)
record_allocator("retain zero")
allocator.release(0)
assert allocator.ref_counts[0] == 1 and 0 not in allocator.free_blocks
record_allocator("release one lease")
allocator.release(0)
record_allocator("release final lease")
assert allocator.allocate(1) == (0,)
record_allocator("reuse zero")
allocator.release(0)
allocator.release(1)
assert allocator.num_free_blocks == 3
record_allocator("all released")


## 3. Reserve capacity, then commit tokens

Here, **B=2 means each physical block has space for two tokens' K/V entries in every model layer**. Four blocks therefore provide eight physical token slots. Request IDs and physical block IDs are separate numbers: request 0 can own blocks 0 and 1; its ID does not restrict it to block 0.

A request needs whole blocks. The number required is `ceil(total_tokens / B)`, which we compute with integer arithmetic as `(total_tokens + B - 1) // B`. Three tokens need two blocks, even though the final slot is initially unused.

The first reservations give us this layout:

| Physical block | Its two slots | Assigned to | Logical token positions |
|---|---|---|---|
| 0 | 0, 1 | Request 0 | 0, 1 |
| 1 | 2, 3 | Request 0 | 2, 3 |
| 2 | 4, 5 | Request 1 | 0, 1 |
| 3 | 6, 7 | Nobody yet | Free |

Request 0's table is `[0, 1]`, so its capacity is four tokens. We initially process only three. Request 1's table is `[2]`, with capacity two. Physical block 3 remains free. Each request's logical positions start at zero, even though their physical slots differ.

The cache constructor allocates the full K and V tensors once, including currently free blocks. Each tensor has shape `(layers, 4, 2, KV_heads, head_dim)`. Reserving blocks updates CPU-side ownership tables; it does not create another GPU tensor. Releasing blocks later will make their slots reusable, not reduce this tensor allocation.

### Capacity and committed length are different

**Capacity** is how many processed tokens the assigned blocks could hold. **Committed length** is how many tokens have successfully completed all model layers and the logits calculation.

| Point in the demo | Request 0 capacity / length | Request 1 capacity / length |
|---|---|---|
| After reservations, before forward | 4 / 0 | 2 / 0 |
| After successful initial forward | 4 / 3 | 2 / 2 |

`reserve(request_id, total_tokens)` takes the **total capacity needed**, not the number of tokens being appended. For example, a request already at length three that will append two tokens needs `reserve(0, 5)`, not `reserve(0, 2)`. Reserving five slots adds enough whole blocks but leaves length three until the forward succeeds. Calling reserve again within existing capacity does not allocate another block.

### What tokens and forwards are we testing?

We tokenize real prompts, then take short slices so the lengths are exactly three and two tokens. Later we append the next tokens from those same fixed prompts. The model is not choosing those appended tokens. This lets us test the cache lifetime without also building a generation loop.

The forward still uses D2's path: metadata supplies request IDs and logical positions; the block table maps those positions to physical slots; attention gathers each request's own K/V prefix. Only requests included in a successful forward advance their committed lengths. An omitted request is paused, not deleted or advanced.

For every newly processed token row, `compare_paged_batch` compares the cached logits with fresh no-cache forwards from our model and Hugging Face. FP32 values may differ slightly because computation order changes. The numerical check allows `abs(actual - expected) <= 1e-4 + 1e-5 * abs(expected)` (`atol=1e-4`, `rtol=1e-5`). A separate check requires identical `argmax` token IDs on **every new row**, not only the final row. Close logits alone do not guarantee identical token choices.

In [ ]:
tokenizer, hf_model = load_model(device=DEVICE, dtype=torch.float32)
hf_model.set_attn_implementation("eager")
our_model = LlamaForCausalLM.from_huggingface(hf_model)
prompt_texts = ["The capital of France is", "The capital of Germany is",
                "A tiny language model can help people by"]
encoded_prompts = [encode(tokenizer, text, DEVICE) for text in prompt_texts]
assert all(ids.numel() >= 5 for ids in encoded_prompts)
BLOCK_SIZE, NUM_BLOCKS = 2, 4
managed_pool = our_model.new_managed_paged_cache(NUM_BLOCKS, BLOCK_SIZE)
managed_pool.key.fill_(float("nan"))
managed_pool.value.fill_(float("nan"))
pool_pointers = (managed_pool.key.data_ptr(), managed_pool.value.data_ptr())
stage_results, pool_trace = [], []

def pool_state():
    return {"tables": {str(owner): list(table) for owner, table in managed_pool.block_tables.items()},
            "lengths": {str(owner): length for owner, length in managed_pool.lengths.items()},
            "free_blocks": list(managed_pool.free_blocks), "ref_counts": list(managed_pool.ref_counts)}

def record_pool(label):
    used = [block for table in managed_pool.block_tables.values() for block in table]
    assert len(used) == len(set(used))
    assert set(used).isdisjoint(managed_pool.free_blocks)
    assert set(used).union(managed_pool.free_blocks) == set(range(NUM_BLOCKS))
    assert managed_pool.ref_counts == tuple(int(block in used) for block in range(NUM_BLOCKS))
    assert pool_pointers == (managed_pool.key.data_ptr(), managed_pool.value.data_ptr())
    row = {"label": label, **pool_state()}
    pool_trace.append(row)
    print(row)

def run_stage(label, active_requests):
    comparison = compare_paged_batch(our_model, hf_model, managed_pool, active_requests)
    stage_results.append({"label": label, "comparison": comparison})
    record_pool(label)
    print(comparison)

assert managed_pool.reserve(0, 3) == (0, 1)
assert managed_pool.reserve(1, 2) == (2,)
assert dict(managed_pool.lengths) == {0: 0, 1: 0}
record_pool("reserved, not committed")
run_stage("initial prefill", {0: encoded_prompts[0][:3], 1: encoded_prompts[1][:2]})
assert dict(managed_pool.lengths) == {0: 3, 1: 2}


## 4. Exhaustion is not permission to steal live blocks

After the initial forward, request 0 owns `[0, 1]` and has committed length three. Request 1 owns `[2]` and has length two. Only block 3 is free. The next cell tries two reservations that cannot fit:

| Attempt | Total blocks needed | Blocks already owned | Additional blocks needed | Free blocks |
|---|---|---|---|---|
| Admit request 2 for four tokens | 2 | 0 | 2 | 1 |
| Grow request 0 to seven tokens | 4 | 2 | 2 | 1 |

Both calls must raise `BlockPoolExhausted`. Here, **atomic failure** means all or nothing: the allocator must not take the one remaining block and then fail halfway through. A failed admission must not add request 2 to the tables. A failed growth must not alter request 0's existing table. Free IDs, reference counts and committed lengths must all match their snapshots from before the call.

We then make a smaller growth request: `reserve(0, 5)`. Five tokens need three blocks; request 0 already owns two. The one remaining free block is sufficient. Its table becomes `[0, 1, 3]`, with capacity six and committed length still three. The pool now has no free blocks.

D3 raises rather than taking blocks from live requests. A future scheduler could wait for space or preempt a request, but that policy is not implemented here. Each individual reserve call is atomic; a sequence of successful reserve calls is not one shared batch transaction.

In [ ]:
exhaustion_results = []
for request_id, total_tokens in ((2, 4), (0, 7)):
    before = pool_state()
    try:
        managed_pool.reserve(request_id, total_tokens)
    except BlockPoolExhausted as error:
        row = {"request_id": request_id, "total_tokens": total_tokens, "error": str(error),
               "state_unchanged": pool_state() == before}
        assert row["state_unchanged"]
        exhaustion_results.append(row)
        print("expected exhaustion:", row)
    else:
        raise AssertionError("insufficient capacity must raise")
assert managed_pool.reserve(0, 5) == (0, 1, 3)
assert managed_pool.lengths[0] == 3 and managed_pool.num_free_blocks == 0
record_pool("grown across boundary, still length three")


## 5. A failed forward keeps its reservation for retry

Request 0 now has capacity six, but only positions 0, 1 and 2 are committed. We want to append positions 3 and 4, bringing its length to five. These positions cross a block boundary: position 3 uses the final slot of physical block 1; position 4 uses the first slot of physical block 3.

To test failure handling, we install a **forward pre-hook** that raises an error just before the second decoder layer runs. The first layer has already written K/V for the new positions, but the remaining layers and logits have not completed. Treating those tokens as committed would expose an incomplete cache on the next forward.

We check two separate promises after the error:

1. Ownership tables, free IDs, reference counts and committed lengths are unchanged. Request 0 keeps its reservation and remains at length three; request 1 remains at length two.
2. The old committed prefix's K **and** V bytes are unchanged in every layer, for both requests. Only uncommitted append slots may contain partial results.

The `finally` block removes the failure hook even if a check fails. We can then retry the same positions 3 and 4. Retry overwrites the partial append data, completes the full forward, and commits request 0's length to five. The parity helper checks every retry logits row against fresh references.

Request 1 does not appear in this forward. Its length stays two: pausing a request means doing no work for it, not releasing its blocks.

In [ ]:
def prefix_bytes(request_id):
    positions = torch.arange(managed_pool.lengths[request_id], device=managed_pool.key.device)
    owners = positions.new_full(positions.shape, request_id)
    slots = managed_pool.slot_mapping(positions, owners).long()
    return tuple(storage.flatten(1, 2)[:, slots].clone()
                 for storage in (managed_pool.key, managed_pool.value))

prefix_before = {owner: prefix_bytes(owner) for owner in managed_pool.lengths}
before_failure = pool_state()

def injected_failure(module, args):
    raise RuntimeError("injected D3 later-layer failure")

failure_hook = our_model.model.layers[1].register_forward_pre_hook(injected_failure)
try:
    try:
        with torch.no_grad():
            append_ids = encoded_prompts[0][3:5]
            append_positions = torch.arange(3, 5, device=append_ids.device, dtype=append_ids.dtype)
            append_owners = torch.zeros_like(append_ids)
            our_model(append_ids, append_positions,
                      AttentionMetadata(append_positions, append_owners, paged_cache=managed_pool))
    except RuntimeError as error:
        if str(error) != "injected D3 later-layer failure":
            raise
        failure_error = str(error)
        print("expected forward failure:", failure_error)
    else:
        raise AssertionError("failure hook did not execute")
finally:
    failure_hook.remove()
assert pool_state() == before_failure
for owner, previous in prefix_before.items():
    for actual, expected in zip(prefix_bytes(owner), previous):
        torch.testing.assert_close(actual, expected, rtol=0, atol=0)
record_pool("failure preserved ownership and committed prefixes")
run_stage("retry boundary-crossing append", {0: encoded_prompts[0][:5]})
assert dict(managed_pool.lengths) == {0: 5, 1: 2}


## 6. Release, reuse and mixed work

Request 0 has finished the work we want to demonstrate. `release_request(0)` removes its table and length, then returns blocks 0, 1 and 3 to the free list. Request 1 still owns block 2 and keeps its committed length of two.

Because the free list is a stack, those released blocks will be allocated in order 3, 1, 0. Request 2 reserves four token slots and receives `[3, 1]`. Request 1 grows to three token slots and receives block 0, extending its table to `[2, 0]`.

| Request | New table | Capacity | Length before the mixed forward |
|---|---|---|---|
| 1 | `[2, 0]` | 4 | 2 |
| 2 | `[3, 1]` | 4 | 0 |

These physical IDs are not in ascending order. That is valid: request 2's logical positions 0, 1, 2, 3 map to physical slots 6, 7, 2, 3. Sorting its table would change the mapping and corrupt token order. D2's addressing must work regardless of which IDs the allocator returns.

The next forward combines two kinds of work into five flat token rows:

- **Prefill:** request 2 processes its first four tokens at logical positions 0 through 3.
- **Append:** request 1 processes one more token at logical position 2, using its existing two-token prefix.

After success, their committed lengths are four and three respectively. Both must match fresh reference forwards. Request 0's old K/V bytes still exist in the reused blocks, but request 2 writes its own entries before attention reads them. Attention must never gather an unwritten tail or another request's prefix.

### Reset is different from release

`reset(2)` sets request 2's committed length to zero but keeps its table `[3, 1]`. No blocks become free. We then process a different two-token prefix for that request; its old four-token prefix must no longer be visible. Request 1 remains unchanged.

Finally, releasing both remaining requests removes their metadata and makes all four blocks free. We check that the K/V tensor addresses never changed throughout the demo. Reset and release both leave the allocated GPU tensors in place, and neither securely erases their old contents.

In [ ]:
released_blocks = set(managed_pool.block_tables[0])
managed_pool.release_request(0)
assert 0 not in managed_pool.block_tables and 0 not in managed_pool.lengths
assert managed_pool.num_free_blocks == 3 and managed_pool.lengths[1] == 2
record_pool("request zero released")
managed_pool.reserve(2, 4)
assert set(managed_pool.block_tables[2]) <= released_blocks
reused_blocks = list(managed_pool.block_tables[2])
managed_pool.reserve(1, 3)
record_pool("reused blocks and grew remaining request")
run_stage("mixed new prefill and resumed append", {2: encoded_prompts[2][:4], 1: encoded_prompts[1][:3]})
assert dict(managed_pool.lengths) == {1: 3, 2: 4}
final_lengths_before_reset = dict(managed_pool.lengths)
reset_table = managed_pool.block_tables[2]
managed_pool.reset(2)
assert managed_pool.lengths[2] == 0 and managed_pool.lengths[1] == 3
assert managed_pool.block_tables[2] == reset_table and managed_pool.num_free_blocks == 0
run_stage("reset without release", {2: encoded_prompts[0][:2]})
for request_id in tuple(managed_pool.block_tables):
    managed_pool.release_request(request_id)
assert managed_pool.num_free_blocks == NUM_BLOCKS and managed_pool.ref_counts == (0,) * NUM_BLOCKS
assert dict(managed_pool.block_tables) == dict(managed_pool.lengths) == {}
record_pool("all requests released")


## 7. Save evidence and interview questions

The result file records allocator states, request tables, committed lengths, expected exhaustion errors, the injected failure, reuse checks and logits comparisons. It also includes the test output and environment information, including the GPU, library versions and Git commit. This lets us distinguish an observed Colab run from code that has only passed local static checks.

The saved numerical ranges summarize the maximum absolute logits difference in each request comparison. They are correctness measurements, not timings or throughput results. D3 does not claim faster attention or a smaller GPU tensor allocation.

Use these questions to check your understanding:

1. **Why count references?** One physical block may have more than one ownership claim. Releasing one claim must not make the block free while another claim remains. The count measures ownership, not stored tokens. Actual managed requests in this lesson remain disjoint.
2. **Why separate reservation and committed length?** Capacity must be available before execution. Tokens become part of the committed prefix only after all layers and logits succeed. A failed forward keeps its reservation so retry can overwrite incomplete append entries.
3. **Does release reduce GPU allocation?** No. It makes block IDs available to other requests inside an already allocated tensor pool. Choosing which request runs next belongs to the scheduler, not the allocator.

Save the JSON and the notebook with successful outputs before replacing the runtime. **Next after D3 validation: D4, continuous batching and preemption.**

In [ ]:
all_rows = [row for stage in stage_results for row in stage["comparison"]["requests"]]
no_cache_diffs = [row["paged_vs_no_cache_max_abs_diff"] for row in all_rows]
hf_diffs = [row["paged_vs_hf_max_abs_diff"] for row in all_rows]
saved_path = save_result("D3_block_manager", {
    "dtype": "float32", "backend": "pytorch_paged_gather", "managed_disjoint_blocks": True,
    "block_size": BLOCK_SIZE, "num_blocks": NUM_BLOCKS,
    "pool_shape": list(managed_pool.key.shape), "pool_allocated_bytes": managed_pool.allocated_bytes,
    "allocator_trace": allocator_trace, "pool_trace": pool_trace, "stages": stage_results,
    "exhaustion": exhaustion_results, "forward_failure_error": failure_error,
    "failed_forward_preserved_metadata_and_prefix_kv": True,
    "reused_blocks": reused_blocks, "final_lengths_before_reset": final_lengths_before_reset,
    "storage_addresses_unchanged": pool_pointers == (managed_pool.key.data_ptr(), managed_pool.value.data_ptr()),
    "all_blocks_released": managed_pool.num_free_blocks == NUM_BLOCKS,
    "paged_vs_no_cache_diff_range": {"min": min(no_cache_diffs), "max": max(no_cache_diffs)},
    "paged_vs_hf_diff_range": {"min": min(hf_diffs), "max": max(hf_diffs)},
    "pytest": {"returncode": test_run.returncode, "stdout": test_run.stdout, "stderr": test_run.stderr},
})
print(saved_path)
print(saved_path.read_text())


In [ ]:
del managed_pool, prefix_before, previous, actual, expected, our_model, hf_model, failure_hook
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory still held by this kernel: {torch.cuda.memory_allocated() / 2**20:.0f} MiB")
